# Training Notebook — Synthetic Shapes Dataset

Generates a synthetic instance segmentation dataset (circles, rectangles, triangles), registers it with Detectron2, and runs training to validate the full pipeline.

In [ ]:
import sys
import json
import os
import random
import math
from pathlib import Path

import cv2
import numpy as np

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from training_scripts.train import register_instances, train

## 1. Generate Synthetic Shapes Dataset

In [ ]:
IMG_W, IMG_H = 256, 256
MIN_RADIUS, MAX_RADIUS = 15, 50
MAX_OBJECTS = 5
CATEGORY = {"id": 1, "name": "circle", "supercategory": "shape"}


def random_color():
    return [random.randint(60, 255) for _ in range(3)]


def generate_image_and_annotations(image_id, ann_id_start):
    img = np.random.randint(20, 80, (IMG_H, IMG_W, 3), dtype=np.uint8)
    annotations = []
    ann_id = ann_id_start
    num_objects = random.randint(1, MAX_OBJECTS)

    for _ in range(num_objects):
        r = random.randint(MIN_RADIUS, MAX_RADIUS)
        cx = random.randint(r + 1, IMG_W - r - 1)
        cy = random.randint(r + 1, IMG_H - r - 1)
        color = random_color()

        cv2.circle(img, (cx, cy), r, color, -1)

        # Build polygon segmentation (circle approximated as polygon)
        angles = np.linspace(0, 2 * np.pi, 32, endpoint=False)
        poly_x = (cx + r * np.cos(angles)).tolist()
        poly_y = (cy + r * np.sin(angles)).tolist()
        seg = []
        for px, py in zip(poly_x, poly_y):
            seg.extend([round(px, 1), round(py, 1)])

        x_min = max(0, cx - r)
        y_min = max(0, cy - r)
        w = min(IMG_W, cx + r) - x_min
        h = min(IMG_H, cy + r) - y_min

        annotations.append({
            "id": ann_id,
            "image_id": image_id,
            "category_id": 1,
            "segmentation": [seg],
            "area": float(np.pi * r * r),
            "bbox": [x_min, y_min, w, h],
            "iscrowd": 0,
        })
        ann_id += 1

    return img, annotations, ann_id


def generate_split(split_dir, num_images):
    os.makedirs(split_dir, exist_ok=True)
    images_info = []
    all_annotations = []
    ann_id = 1

    for i in range(num_images):
        filename = f"{i:04d}.png"
        img, anns, ann_id = generate_image_and_annotations(i, ann_id)
        cv2.imwrite(os.path.join(split_dir, filename), img)

        images_info.append({
            "id": i,
            "file_name": filename,
            "width": IMG_W,
            "height": IMG_H,
        })
        all_annotations.extend(anns)

    coco = {
        "images": images_info,
        "annotations": all_annotations,
        "categories": [CATEGORY],
    }

    split_name = os.path.basename(split_dir)
    json_path = os.path.join(split_dir, f"{split_name}.json")
    with open(json_path, "w") as f:
        json.dump(coco, f)

    print(f"  {split_name}: {num_images} images, {len(all_annotations)} instances -> {json_path}")

In [ ]:
DATASET_DIR = str(PROJECT_ROOT / "data" / "toy")

random.seed(42)
np.random.seed(42)

print("Generating toy dataset...")
generate_split(os.path.join(DATASET_DIR, "train"), num_images=30)
generate_split(os.path.join(DATASET_DIR, "val"), num_images=8)
generate_split(os.path.join(DATASET_DIR, "test"), num_images=8)
print("Done!")

## 2. Register Dataset & Train

In [ ]:
from detectron2 import model_zoo

# Register the synthetic dataset splits
register_instances(DATASET_DIR)

BASE_MODEL = "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"

config = {
    "BASE_MODEL": BASE_MODEL,
    "JOBNAME": "toy_circle_test",
    "EVAL_PERIOD": 100,

    "MODEL": {
        "WEIGHTS": model_zoo.get_checkpoint_url(BASE_MODEL),
        "ROI_HEADS": {
            "NUM_CLASSES": 1,
            "BATCH_SIZE_PER_IMAGE": 128,
            "SCORE_THRESH_TEST": 0.3,
        },
    },

    "DATASETS": {
        "TRAIN": ["my_dataset_train"],
        "TEST": ["my_dataset_test", "my_dataset_val"],
    },

    "DATALOADER": {
        "NUM_WORKERS": 2,
        "FILTER_EMPTY_ANNOTATIONS": False,
    },

    "SOLVER": {
        "MAX_ITER": 300,
        "IMS_PER_BATCH": 2,
        "BASE_LR": 0.001,
        "STEPS": [200, 250],
        "GAMMA": 0.5,
        "WARMUP_ITERS": 50,
        "WARMUP_FACTOR": 0.1,
    },

    "OUTPUT_DIR": "model_output/",
}

train(config)